## 자막 퀄리티 조사용 2025-10-02 00:29

마침표가 없다.
자동 자막이라 중간에 오타나 부자연스러운 단어가 있을 가능성이 있다.

In [2]:
from urllib.parse import urlparse, parse_qs

def extract_video_id(url: str) -> str | None:
    """
    YouTube URL에서 videoId를 추출.
    지원 예시:
     - https://www.youtube.com/watch?v=VIDEO_ID
     - https://youtu.be/VIDEO_ID
     - https://www.youtube.com/embed/VIDEO_ID
     - https://www.youtube.com/v/VIDEO_ID
    """
    parsed = urlparse(url)
    host = parsed.netloc
    path = parsed.path

    # 예: youtu.be/VIDEO_ID
    if host in ("youtu.be", "www.youtu.be"):
        # path 시작 "/" 이후 부분이 ID
        return path.lstrip("/")

    # 예: www.youtube.com/watch?v=VIDEO_ID
    if host in ("www.youtube.com", "youtube.com", "m.youtube.com"):
        # 쿼리 파라미터 중 v 값을 봄
        qs = parse_qs(parsed.query)
        if "v" in qs:
            return qs["v"][0]

        # embed 형식 등
        # /embed/VIDEO_ID
        if path.startswith("/embed/"):
            return path.split("/")[2]
        # /v/VIDEO_ID
        if path.startswith("/v/"):
            return path.split("/")[2]

    return None  # ID를 못 찾았을 경우

In [3]:
from youtube_transcript_api import YouTubeTranscriptApi

video_id = extract_video_id("https://www.youtube.com/watch?v=EWvNQjAaOHw&t=6026s")

ytt_api = YouTubeTranscriptApi()
youtube_transcripts_data = ytt_api.fetch(video_id)

In [4]:
youtube_transcripts_data

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='hi everyone so in this video I would', start=0.12, duration=3.72), FetchedTranscriptSnippet(text='like to continue our general audience', start=2.159, duration=5.041), FetchedTranscriptSnippet(text='series on large language models like', start=3.84, duration=5.839), FetchedTranscriptSnippet(text='chpd now in the previous video deep dive', start=7.2, duration=3.88), FetchedTranscriptSnippet(text='into llms that you can find on my', start=9.679, duration=3.281), FetchedTranscriptSnippet(text='YouTube we went into a lot of the', start=11.08, duration=3.2), FetchedTranscriptSnippet(text='underhood fundamentals of how these', start=12.96, duration=3.2), FetchedTranscriptSnippet(text='models are trained and how you should', start=14.28, duration=3.8), FetchedTranscriptSnippet(text='think about their cognition or', start=16.16, duration=4.84), FetchedTranscriptSnippet(text='psychology now in this video I want to', start=18.08, duratio

In [5]:
Full_text = ""
for data in youtube_transcripts_data.snippets:
    Full_text += data.text + " " 

In [6]:
Full_text[:5000]

"hi everyone so in this video I would like to continue our general audience series on large language models like chpd now in the previous video deep dive into llms that you can find on my YouTube we went into a lot of the underhood fundamentals of how these models are trained and how you should think about their cognition or psychology now in this video I want to go into more practical applications of these tools I want to show you lots of examples I want to take you through all the different settings that are available and I want to show you how I use these tools and how you can also use them uh in your own life and work so let's dive in okay so first of all the web page that I have pulled up here is chp.com now as you might know chpt it was developed by openai and deployed in 2022 so this was the first time that people could actually just kind of like talk to a large language model through a text interface and this went viral and over all over the place on the internet and uh this wa

In [6]:
from youtube_transcript_api.formatters import TextFormatter

text_formatter = TextFormatter()
print(text_formatter.format_transcript(youtube_transcripts_data))

hi everyone so in this video I would
like to continue our general audience
series on large language models like
chpd now in the previous video deep dive
into llms that you can find on my
YouTube we went into a lot of the
underhood fundamentals of how these
models are trained and how you should
think about their cognition or
psychology now in this video I want to
go into more practical applications of
these tools I want to show you lots of
examples I want to take you through all
the different settings that are
available and I want to show you how I
use these tools and how you can also use
them uh in your own life and work so
let's dive in okay so first of all the
web page that I have pulled up here is
chp.com now as you might know chpt it
was developed by openai and deployed in
2022 so this was the first time that
people could actually just kind of like
talk to a large language model through a
text interface and this went viral and
over all over the place on the internet
and uh this was

In [13]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [23]:
# pip install wtpsplit regex nltk
from wtpsplit import SaT
import re

DISCOURSE = r"(?:okay|so|now|first of all|in this video|for example|in particular|since then|however|therefore|as a result)"
DISCOURSE_RE = re.compile(rf"\b{DISCOURSE}\b", re.IGNORECASE)

def segment(text: str, method="sat", threshold=0.3):
    txt = (text)
    seg = SaT("sat-3l-sm")  # 작고 빠른 모델
    raw = seg.split(txt, threshold=threshold)
    return (raw)

# 사용
sents = segment(Full_text, method="sat", threshold=0.35)


In [35]:
list(map(lambda x: x if len(x)>=10 else None, sents))

['hi everyone ',
 'so in this video I would like to continue our general audience series on large language models like chpd now in the previous video deep dive into llms that you can find on my YouTube ',
 'we went into a lot of the underhood fundamentals of how these models are trained and how you should think about their cognition or psychology ',
 'now in this video I want to go into more practical applications of these tools ',
 'I want to show you lots of examples ',
 'I want to take you through all the different settings that are available and I want to show you how I use these tools and how you can also use them ',
 "uh in your own life and work so let's dive in okay ",
 'so first of all the web page that I have pulled up here is chp.com now as you might know chpt it was developed by openai and deployed in 2022 ',
 'so this was the first time that people could actually just kind of like talk to a large language model through a text interface and this went viral and over all over